# Лабораторная работа №7: Генетические алгоритмы с вещественным и перестановочным представлением особей

**Дисциплина:** Генетические и эволюционные вычисления · **Вариант 3**

Работа объединяет две части:
- **Часть А — ГА с вещественным кодированием:** поиск минимума функции Растригина $f(x_1,x_2) = 20 + \sum_{i=1}^{2}\bigl[x_i^2 - 10\cos(2\pi x_i)\bigr]$, $x_i \in [-5{,}12;\ 5{,}12]$; операторы BLX-α, гауссова мутация, турнирная селекция, элитизм; сравнение с бинарным кодированием из ЛР №6.
- **Часть Б — задача коммивояжёра:** тестовый набор **eil76** (TSPLIB, 76 городов, оптимум 538), представление путей, **циклический кроссинговер (CX)**, мутация обменом и инверсией, локальный поиск 2-opt.

**Цель работы:** изучить представление особей в форме вещественных чисел и перестановок, специализированные генетические операторы для каждого представления и особенности решения задач непрерывной и комбинаторной оптимизации генетическими алгоритмами.

Ноутбук рассчитан на перезапуск по ячейкам в Docker-окружении репозитория (`make up`, http://localhost:8888). Полный прогон — около 3 минут.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

OUTPUTS = Path('report/outputs')
FIGURES = Path('report/figures')
TABLES = Path('report/tables')
for p in (OUTPUTS, FIGURES, TABLES):
    p.mkdir(parents=True, exist_ok=True)

# Палитра проверена на различимость при дальтонизме
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
BLUE_RAMP = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']
INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
GOAL = '#d03b3b'

axis_style = dict(gridcolor=GRID, linecolor=AXIS, zeroline=False, ticks='outside', tickcolor=AXIS, ticklen=4,
                  title_font=dict(size=14, color=INK2), tickfont=dict(size=12, color=INK2))
pio.templates['lab'] = go.layout.Template(layout=dict(
    font=dict(family='system-ui, -apple-system, "Segoe UI", Roboto, Arial, sans-serif', size=14, color=INK),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, colorway=PALETTE,
    xaxis=axis_style, yaxis=axis_style,
    title=dict(x=0.01, xanchor='left', font=dict(size=18, color=INK)),
    legend=dict(bgcolor='rgba(252,252,251,0.85)', bordercolor=GRID, borderwidth=1, font=dict(size=12)),
    hoverlabel=dict(bgcolor='white', bordercolor=AXIS, font=dict(size=12, color=INK)),
    margin=dict(l=70, r=30, t=80, b=60),
))
pio.templates.default = 'plotly_white+lab'
PLOT_CONFIG = dict(displaylogo=False, responsive=True, toImageButtonOptions=dict(format='png', scale=2))


def save(fig, name, width=1000, height=520, png=True, png_fig=None):
    """HTML (интерактивный) в report/outputs и PNG (для отчёта) в report/figures."""
    fig.write_html(OUTPUTS / f'{name}.html', include_plotlyjs='cdn', config=PLOT_CONFIG, auto_play=False)
    if png:
        (png_fig or fig).write_image(FIGURES / f'{name}.png', width=width, height=height, scale=2)


def to_latex(df, name, caption, label, col_format=None):
    """Таблица результатов для отчёта: числа в отчёте совпадают с ноутбуком."""
    body = df.to_latex(index=False, escape=False, column_format=col_format or 'c' * len(df.columns))
    tex = ('\\begin{table}[H]\n\\caption{' + caption + '}\n\\label{' + label + '}\n\\centering\n\\small\n'
           + body + '\\end{table}\n')
    (TABLES / f'{name}.tex').write_text(tex, encoding='utf-8')


def anim_controls(steps, prefix='поколение: ', duration=250):
    """Кнопки «Старт / Пауза / Сброс» и слайдер для анимации Plotly."""
    play = dict(frame=dict(duration=duration, redraw=False), transition=dict(duration=int(duration * 0.8)),
                fromcurrent=True, mode='immediate')
    still = dict(frame=dict(duration=0, redraw=False), mode='immediate', transition=dict(duration=0))
    return dict(
        updatemenus=[dict(type='buttons', direction='left', x=1.0, xanchor='right', y=1.1, yanchor='bottom',
                          showactive=False, buttons=[
                              dict(label='▶ Старт', method='animate', args=[None, play]),
                              dict(label='⏸ Пауза', method='animate', args=[[None], still]),
                              dict(label='⟲ Сброс', method='animate', args=[[str(steps[0])], still])])],
        sliders=[dict(active=0, x=0, len=1, y=-0.08, yanchor='top', pad=dict(t=30),
                      currentvalue=dict(prefix=prefix, font=dict(size=13, color=INK2)),
                      steps=[dict(label=str(s), method='animate', args=[[str(s)], still]) for s in steps])])


print('numpy', np.__version__, '| pandas', pd.__version__, '| plotly', plotly.__version__)

numpy 2.4.4 | pandas 3.0.2 | plotly 6.7.0


# Часть А. ГА с вещественным кодированием

## А.1. Функция Растригина

В области поиска около 100 локальных минимумов, расположенных в узлах целочисленной решётки; глобальный минимум $f(0, 0) = 0$, ближайшие локальные — $f \approx 0{,}995$.

In [2]:
BOUNDS = np.array([[-5.12, 5.12], [-5.12, 5.12]])


def rastrigin(X):
    X = np.atleast_2d(X)
    return 20 + np.sum(X ** 2 - 10 * np.cos(2 * np.pi * X), axis=1)


def schaffer_f7(X):
    X = np.atleast_2d(X)
    r2 = np.sum(X ** 2, axis=1)
    return r2 ** 0.25 * (np.sin(50 * r2 ** 0.1) ** 2 + 1)


g = np.linspace(-5.12, 5.12, 201)
G1, G2 = np.meshgrid(g, g)
Z_R = rastrigin(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)
print(f'Локальных минимумов в узлах решётки: {sum(1 for a in range(-5, 6) for b in range(-5, 6))}; '
      f'f(1, 0) = {rastrigin([1, 0])[0]:.3f}')

Локальных минимумов в узлах решётки: 121; f(1, 0) = 1.000


## А.2. Реализация

- **Представление:** особь — вектор $(x_1, x_2) \in \mathbb R^2$.
- **Турнирная селекция** ($k = 3$).
- **BLX-α кроссинговер:** $c_i \sim U[\min(p_i^1, p_i^2) - \alpha d_i;\ \max(p_i^1, p_i^2) + \alpha d_i]$, $d_i = |p_i^1 - p_i^2|$. При $\alpha = 0$ потомок лежит между родителями (популяция сжимается), при $\alpha \approx 0{,}5$ интервал поиска сохраняется.
- **Гауссова мутация** с вероятностью $P_m$ на ген: $x_i' = x_i + \mathcal N(0, \sigma(t))$; шаг $\sigma(t) = \sigma_0 (b - a)\,(1 - t/T) + 10^{-4}$ уменьшается к концу поиска — сначала исследование, затем точная доводка.
- **Элитизм:** 2 лучшие особи переходят в следующее поколение без изменений.

Успех запуска — найден глобальный минимум с $f < 10^{-3}$.

In [3]:
DEF_R = dict(pop_size=60, n_gen=150, p_c=0.9, p_m=0.2, alpha=0.5, sigma0=0.1, k=3, elite=2)


def real_ga(func=rastrigin, bounds=BOUNDS, pop_size=60, n_gen=150, p_c=0.9, p_m=0.2, alpha=0.5, sigma0=0.1, k=3,
            elite=2, seed=0, track=False):
    rng = np.random.default_rng(seed)
    lo, hi = bounds[:, 0], bounds[:, 1]
    P = rng.uniform(lo, hi, (pop_size, len(lo)))
    F = func(P)
    hb, hm, snaps = [F.min()], [np.median(F)], [P.copy()] if track else None
    for t in range(n_gen):
        order = np.argsort(F)
        elites = P[order[:elite]].copy()
        cand = rng.integers(0, pop_size, (pop_size, k))
        par = P[cand[np.arange(pop_size), np.argmin(F[cand], axis=1)]]
        p1, p2 = par[0::2], par[1::2]
        d = np.abs(p1 - p2)
        lo_i, hi_i = np.minimum(p1, p2) - alpha * d, np.maximum(p1, p2) + alpha * d
        c1, c2 = rng.uniform(lo_i, hi_i), rng.uniform(lo_i, hi_i)
        keep = rng.random(len(p1)) >= p_c
        c1[keep], c2[keep] = p1[keep], p2[keep]
        kids = np.vstack([c1, c2])
        sigma = sigma0 * (hi - lo) * (1 - t / n_gen) + 1e-4
        m = rng.random(kids.shape) < p_m
        kids = np.clip(kids + m * rng.normal(0, 1, kids.shape) * sigma, lo, hi)
        kids[:elite] = elites
        P, F = kids, func(kids)
        hb.append(F.min()); hm.append(np.median(F))
        if track:
            snaps.append(P.copy())
    i = int(np.argmin(F))
    return dict(x=P[i], f=F[i], hist_best=np.array(hb), hist_med=np.array(hm), snaps=snaps)


res_r = real_ga(**DEF_R, seed=0, track=True)
print(f'Растригин: x = ({res_r["x"][0]:+.2e}; {res_r["x"][1]:+.2e}), f = {res_r["f"]:.2e} '
      f'({DEF_R["pop_size"] * (DEF_R["n_gen"] + 1)} вычислений функции)')

Растригин: x = (+1.02e-09; -1.42e-09), f = 0.00e+00 (9060 вычислений функции)


In [4]:
# Популяция на линиях уровня: анимация (Старт / Пауза / Сброс) и раскадровка для отчёта
steps = [0, 3, 6, 10, 15, 20, 30, 50, 80, 150]
fig = go.Figure()
fig.add_trace(go.Contour(x=g, y=g, z=Z_R, colorscale='Blues', reversescale=True, ncontours=20, showscale=False,
                         line=dict(width=0.3), hoverinfo='skip'))
fig.add_trace(go.Scatter(x=res_r['snaps'][0][:, 0], y=res_r['snaps'][0][:, 1], mode='markers', name='популяция',
                         marker=dict(color=PALETTE[1], size=8, line=dict(color=SURFACE, width=1))))
fig.frames = [go.Frame(name=str(s), data=[go.Scatter(x=res_r['snaps'][s][:, 0], y=res_r['snaps'][s][:, 1])], traces=[1],
                       layout=go.Layout(title_text=f'Растригин: популяция — поколение {s}')) for s in steps]
fig.update_layout(title='Растригин: популяция — поколение 0', xaxis_title='x₁', yaxis_title='x₂', width=720, height=740,
                  margin=dict(t=110, b=120), yaxis=dict(scaleanchor='x'), **anim_controls(steps))
png = make_subplots(rows=1, cols=3, horizontal_spacing=0.05, subplot_titles=[f'поколение {s}' for s in (0, 10, 150)])
for c, s in enumerate((0, 10, 150), start=1):
    png.add_trace(go.Contour(x=g, y=g, z=Z_R, colorscale='Blues', reversescale=True, ncontours=20, showscale=False,
                             line=dict(width=0.3)), 1, c)
    png.add_trace(go.Scatter(x=res_r['snaps'][s][:, 0], y=res_r['snaps'][s][:, 1], mode='markers', showlegend=False,
                             marker=dict(color=PALETTE[1], size=7, line=dict(color=SURFACE, width=1))), 1, c)
    png.update_yaxes(scaleanchor=f'x{c if c > 1 else ""}', row=1, col=c)
png.update_layout(title='Вещественный ГА на функции Растригина', height=440)
save(fig, 'A1_population', width=1250, height=440, png_fig=png)
fig.show()

## А.3. Параметры вещественного ГА (20 запусков на значение)

При 150 поколениях почти любая настройка находит глобальный минимум, поэтому для сравнения бюджет сокращён до **50 поколений** (3060 вычислений функции) — так различия между настройками становятся видны.

In [5]:
R = 20


def real_batch(**kw):
    rr = [real_ga(**{**DEF_R, 'n_gen': 50, **kw}, seed=300 + s) for s in range(R)]
    fb = np.array([r['f'] for r in rr])
    return dict(медиана_f=np.median(fb), худшее_f=fb.max(), успехов=np.mean(fb < 1e-3) * 100), rr


GRID_R = {'alpha': [0.0, 0.25, 0.5, 1.0], 'sigma0': [0.0, 0.02, 0.1, 0.3], 'p_m': [0.0, 0.05, 0.2, 0.5],
          'pop_size': [20, 40, 60, 120]}
LAB_R = {'alpha': 'параметр BLX-α', 'sigma0': 'шаг мутации σ₀', 'p_m': 'вероятность мутации Pm', 'pop_size': 'размер популяции N'}
rows = []
tic = time.perf_counter()
for par, vals in GRID_R.items():
    for v in vals:
        rows.append(dict(параметр=LAB_R[par], par=par, значение=v, **real_batch(**{par: v})[0]))
df_r = pd.DataFrame(rows)
print(f'Готово за {time.perf_counter() - tic:.1f} с')
df_r.drop(columns='par').style.format({'медиана_f': '{:.1e}', 'худшее_f': '{:.2f}', 'успехов': '{:.0f} %'}).hide(axis='index')

Готово за 0.8 с


параметр,значение,медиана_f,худшее_f,успехов
параметр BLX-α,0.000000,6.5e-09,0.00,100 %
параметр BLX-α,0.250000,1.8e-12,0.00,100 %
параметр BLX-α,0.500000,2.1e-12,0.00,100 %
параметр BLX-α,1.000000,2.6e-08,0.99,95 %
шаг мутации σ₀,0.000000,0.0e+00,0.99,80 %
шаг мутации σ₀,0.020000,2.8e-12,0.99,70 %
шаг мутации σ₀,0.100000,2.1e-12,0.00,100 %
шаг мутации σ₀,0.300000,7.5e-12,0.00,100 %
вероятность мутации Pm,0.000000,0.0e+00,0.99,80 %
вероятность мутации Pm,0.050000,0.0e+00,0.99,85 %


In [6]:
fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.03, subplot_titles=[LAB_R[p] for p in GRID_R])
for c, par in enumerate(GRID_R, start=1):
    d = df_r[df_r.par == par]
    fig.add_trace(go.Bar(x=[f'{v:g}' for v in d.значение], y=d.успехов, showlegend=False,
                         marker=dict(color=PALETTE[0], line=dict(color=SURFACE, width=2)),
                         text=[f'{v:.0f}%' for v in d.успехов], textposition='outside', textfont=dict(size=11, color=INK2),
                         customdata=np.c_[d.медиана_f, d.худшее_f],
                         hovertemplate='%{x}: успехов %{y:.0f}%<br>медиана f = %{customdata[0]:.1e}<br>'
                                       'худшее f = %{customdata[1]:.2f}<extra></extra>'), 1, c)
fig.update_xaxes(type='category')
fig.update_yaxes(range=[0, 115], title_text='доля запусков с f < 10⁻³, %', row=1, col=1)
fig.update_layout(title='Растригин, бюджет 50 поколений: доля запусков, нашедших глобальный минимум (20 запусков)', height=420, bargap=0.3)
save(fig, 'A2_parameters', width=1250, height=420)
fig.show()

## А.4. Вещественное против бинарного кодирования

Бинарный ГА из ЛР №6 (20 бит на переменную, турнир, однородный кроссинговер, побитовая мутация, элитарное сокращение) и вещественный ГА получают одинаковый бюджет — 60 особей × 150 поколений. Сравниваются две функции: Растригина и Шаффера F7 (область $[-100;\ 100]^2$).

In [7]:
def binary_ga(func, bounds, pop_size=60, n_gen=150, p_c=0.9, L=20, k=3, seed=0):
    rng = np.random.default_rng(seed)
    lo, hi = bounds[:, 0], bounds[:, 1]
    w = 2 ** np.arange(L - 1, -1, -1)

    def dec(P):
        return np.column_stack([lo[j] + (P[:, j * L:(j + 1) * L] @ w) * (hi[j] - lo[j]) / (2 ** L - 1) for j in range(len(lo))])

    n = L * len(lo)
    P = rng.integers(0, 2, (pop_size, n), dtype=np.int8)
    F = func(dec(P)); hb = [F.min()]
    for _ in range(n_gen):
        cand = rng.integers(0, pop_size, (pop_size, k))
        par = P[cand[np.arange(pop_size), np.argmin(F[cand], axis=1)]]
        p1, p2 = par[0::2], par[1::2]
        m = rng.integers(0, 2, p1.shape).astype(bool) & (rng.random(len(p1)) < p_c)[:, None]
        K = np.vstack([np.where(m, p2, p1), np.where(m, p1, p2)])
        K ^= (rng.random(K.shape) < 1 / n).astype(np.int8)
        KF = func(dec(K))
        allP, allF = np.vstack([P, K]), np.r_[F, KF]
        o = np.argsort(allF)[:pop_size]
        P, F = allP[o], allF[o]; hb.append(F.min())
    return dict(f=F.min(), hist_best=np.array(hb))


B_F7 = np.array([[-100.0, 100.0], [-100.0, 100.0]])
cmp_rows, cmp_curves = [], {}
for fname, func, bnd in [('Растригин', rastrigin, BOUNDS), ('Шаффер F7', schaffer_f7, B_F7)]:
    for enc, runner in [('бинарное (20 бит)', lambda s: binary_ga(func, bnd, seed=700 + s)),
                        ('вещественное', lambda s: real_ga(func, bnd, **{**DEF_R}, seed=700 + s))]:
        rr = [runner(s) for s in range(R)]
        fb = np.array([r['f'] for r in rr])
        cmp_rows.append(dict(функция=fname, кодирование=enc, медиана_f=np.median(fb), лучшее_f=fb.min(), худшее_f=fb.max()))
        cmp_curves[(fname, enc)] = np.median([r['hist_best'] for r in rr], axis=0)
df_cmp = pd.DataFrame(cmp_rows)
df_cmp.style.format({'медиана_f': '{:.2e}', 'лучшее_f': '{:.2e}', 'худшее_f': '{:.2e}'}).hide(axis='index')

функция,кодирование,медиана_f,лучшее_f,худшее_f
Растригин,бинарное (20 бит),9.46e-09,9.46e-09,1.24e+00
Растригин,вещественное,0.00e+00,0.00e+00,4.64e-11
Шаффер F7,бинарное (20 бит),2.00e-02,2.00e-02,1.79e-01
Шаффер F7,вещественное,1.09e-08,8.27e-13,1.51e-02


In [8]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=('Растригин', 'Шаффер F7'))
for c, fname in enumerate(['Растригин', 'Шаффер F7'], start=1):
    for k, enc in enumerate(['бинарное (20 бит)', 'вещественное']):
        cur = cmp_curves[(fname, enc)]
        fig.add_trace(go.Scatter(x=np.arange(len(cur)), y=np.maximum(cur, 1e-12), mode='lines', name=enc, legendgroup=enc,
                                 showlegend=c == 1, line=dict(color=PALETTE[k], width=2.5)), 1, c)
    fig.update_xaxes(title_text='поколение', row=1, col=c)
fig.update_yaxes(type='log', title_text='лучшее f (медиана 20 запусков)', row=1, col=1)
fig.update_yaxes(type='log', row=1, col=2)
fig.update_layout(title='Вещественное кодирование против бинарного при равном бюджете (60 × 150)', height=440,
                  legend=dict(x=0.01, y=0.02, yanchor='bottom'))
save(fig, 'A3_real_vs_binary', width=1250, height=440)
fig.show()

# Часть Б. Задача коммивояжёра (eil76, циклический кроссинговер)

## Б.1. Данные

Набор **eil76** из TSPLIB: 76 городов, расстояние — евклидово, округлённое до целого (`EUC_2D`: $d_{ij} = \lfloor\sqrt{\Delta x^2 + \Delta y^2} + 0{,}5\rfloor$). Вместе с набором загружается опубликованный оптимальный тур `eil76.opt.tour`; его длина 538 проверяется расчётом.

> При проверке выяснилось, что в прежней версии файла `eil76.tsp` координаты городов 59–76 не совпадали с TSPLIB — на таких данных сравнение с оптимумом 538 некорректно. Файл заменён эталонным.

In [9]:
def read_tsp(path):
    pts = [l.split() for l in open(path) if l.split() and l.split()[0].isdigit()]
    return np.array([[float(p[1]), float(p[2])] for p in pts])


COORDS = read_tsp('data/eil76.tsp')
N_CITY = len(COORDS)
DIST = np.floor(np.hypot(*(COORDS[:, None] - COORDS[None]).transpose(2, 0, 1)) + 0.5)
OPT_TOUR = np.array([int(l.split()[0]) - 1 for l in open('data/eil76.opt.tour') if l.split() and l.split()[0].isdigit()])


def tour_len(t):
    return DIST[t, np.roll(t, -1)].sum()


def pop_len(P):
    return DIST[P, np.roll(P, -1, axis=1)].sum(axis=1)


OPT_LEN = tour_len(OPT_TOUR)
rng0 = np.random.default_rng(0)
rand_len = np.mean([tour_len(rng0.permutation(N_CITY)) for _ in range(1000)])
print(f'Городов: {N_CITY}; длина опубликованного оптимального тура: {OPT_LEN:.0f}; средний случайный тур: {rand_len:.0f}')

Городов: 76; длина опубликованного оптимального тура: 538; средний случайный тур: 2520


## Б.2. Операторы

- **Представление путей:** тур — перестановка номеров городов.
- **Циклический кроссинговер (CX)** (вариант 3): позиции разбиваются на циклы соответствия родителей; в потомке каждый город стоит на той же позиции, что и у одного из родителей (циклы берутся попеременно).
- **Мутация обменом (swap):** меняются местами два города; **инверсия:** переворачивается отрезок тура — меняются только два ребра.
- **2-opt** (локальный поиск): пока есть пара рёбер $(a, b), (c, d)$ с $d_{ac} + d_{bd} < d_{ab} + d_{cd}$, отрезок между ними переворачивается. Реализован векторно: выигрыш всех $O(n^2)$ пар считается за одну операцию NumPy.

In [10]:
def cx(p1, p2):
    """Циклический кроссинговер: циклы позиций берутся попеременно от родителей."""
    n = len(p1)
    pos1 = np.empty(n, int); pos1[p1] = np.arange(n)
    c1, c2 = np.empty(n, int), np.empty(n, int)
    seen = np.zeros(n, bool)
    cyc = 0
    for start in range(n):
        if seen[start]:
            continue
        i = start
        while not seen[i]:
            seen[i] = True
            c1[i], c2[i] = (p1[i], p2[i]) if cyc % 2 == 0 else (p2[i], p1[i])
            i = pos1[p2[i]]
        cyc += 1
    return c1, c2


# пример из теории: P1 = (1 2 3 4 5 6 7 8), P2 = (8 5 2 1 3 6 4 7)
e1, e2 = np.array([1, 2, 3, 4, 5, 6, 7, 8]) - 1, np.array([8, 5, 2, 1, 3, 6, 4, 7]) - 1
k1, k2 = cx(e1, e2)
print('CX: P1 = (1 2 3 4 5 6 7 8), P2 = (8 5 2 1 3 6 4 7) → C1 =', (k1 + 1).tolist(), ' C2 =', (k2 + 1).tolist())

_I, _J = np.triu_indices(N_CITY, 2)
_m = ~((_I == 0) & (_J == N_CITY - 1))
_I, _J = _I[_m], _J[_m]


def two_opt(t):
    t = t.copy()
    while True:
        a, b, c, d = t[_I], t[_I + 1], t[_J], t[(_J + 1) % N_CITY]
        delta = DIST[a, c] + DIST[b, d] - DIST[a, b] - DIST[c, d]
        i = int(np.argmin(delta))
        if delta[i] > -1e-9:
            return t
        t[_I[i] + 1:_J[i] + 1] = t[_I[i] + 1:_J[i] + 1][::-1].copy()

CX: P1 = (1 2 3 4 5 6 7 8), P2 = (8 5 2 1 3 6 4 7) → C1 = [1, 5, 2, 4, 3, 6, 7, 8]  C2 = [8, 2, 3, 1, 5, 6, 4, 7]


## Б.3. Генетический алгоритм

Турнирная селекция ($k = 3$), CX с вероятностью $P_c = 0{,}9$, мутация с вероятностью $P_m$ на особь, элитарное сокращение ($\mu + \lambda$) с удалением дубликатов. Сравниваются:
1. **ГА с CX + swap** (классическая постановка варианта), 150 особей × 500 поколений;
2. **ГА с CX + инверсия** — тот же ГА с мутацией, сохраняющей смежность городов;
3. **меметический ГА:** CX + инверсия + 2-opt для каждого потомка, 60 особей × 60 поколений;
4. **мультистарт 2-opt** без ГА с тем же временем работы — чтобы проверить, что ГА вносит вклад сверх локального поиска.

In [11]:
def tsp_ga(pop_size=150, n_gen=500, p_c=0.9, p_m=0.2, mut='swap', local=False, k=3, seed=0, snap_every=None):
    rng = np.random.default_rng(seed)
    P = np.array([rng.permutation(N_CITY) for _ in range(pop_size)])
    if local:
        P = np.array([two_opt(t) for t in P])
    F = pop_len(P)
    hb, hm, snaps = [F.min()], [F.mean()], {0: P[np.argmin(F)].copy()}
    for gen in range(1, n_gen + 1):
        cand = rng.integers(0, pop_size, (pop_size, k))
        par = P[cand[np.arange(pop_size), np.argmin(F[cand], axis=1)]]
        kids = []
        for i in range(0, pop_size, 2):
            a, b = par[i], par[i + 1]
            if rng.random() < p_c:
                a, b = cx(a, b)
            kids += [a.copy(), b.copy()]
        K = np.array(kids)
        for i in np.where(rng.random(pop_size) < p_m)[0]:
            x, y = np.sort(rng.choice(N_CITY, 2, replace=False))
            if mut == 'swap':
                K[i, [x, y]] = K[i, [y, x]]
            else:
                K[i, x:y + 1] = K[i, x:y + 1][::-1].copy()
        if local:
            K = np.array([two_opt(t) for t in K])
        KF = pop_len(K)
        allP, allF = np.vstack([P, K]), np.r_[F, KF]
        _, u = np.unique(allP, axis=0, return_index=True)
        order = u[np.argsort(allF[u])][:pop_size]
        if len(order) < pop_size:
            order = np.r_[order, np.argsort(allF)[:pop_size - len(order)]]
        P, F = allP[order], allF[order]
        hb.append(F.min()); hm.append(F.mean())
        if snap_every and (gen % snap_every == 0 or gen == n_gen):
            snaps[gen] = P[0].copy()
    return dict(best=P[0].copy(), length=F.min(), hist_best=np.array(hb), hist_mean=np.array(hm), snaps=snaps)


tic = time.perf_counter()
ga_runs = {}
for name, kw, nr in [('ГА: CX + swap', dict(mut='swap'), 10), ('ГА: CX + инверсия', dict(mut='inv'), 10),
                     ('меметический ГА: CX + инверсия + 2-opt', dict(mut='inv', local=True, pop_size=60, n_gen=60), 5)]:
    t0 = time.perf_counter()
    ga_runs[name] = [tsp_ga(**kw, seed=s, snap_every=5 if 'мем' in name else 25) for s in range(nr)]
    ga_runs[name + '_time'] = (time.perf_counter() - t0) / nr
budget = ga_runs['меметический ГА: CX + инверсия + 2-opt_time']
ms_best = []
for s in range(5):
    rng = np.random.default_rng(900 + s)
    t0, best = time.perf_counter(), np.inf
    while time.perf_counter() - t0 < budget:
        best = min(best, tour_len(two_opt(rng.permutation(N_CITY))))
    ms_best.append(best)
print(f'Готово за {time.perf_counter() - tic:.0f} с; бюджет мультистарта 2-opt = {budget:.1f} с на запуск')

rows = []
for name in ['ГА: CX + swap', 'ГА: CX + инверсия', 'меметический ГА: CX + инверсия + 2-opt']:
    L_ = np.array([r['length'] for r in ga_runs[name]])
    rows.append(dict(метод=name, запусков=len(L_), лучший=L_.min(), медиана=np.median(L_), худший=L_.max(),
                     отклонение=100 * (np.median(L_) - OPT_LEN) / OPT_LEN, время=ga_runs[name + '_time']))
L_ = np.array(ms_best)
rows.append(dict(метод='мультистарт 2-opt (без ГА)', запусков=5, лучший=L_.min(), медиана=np.median(L_), худший=L_.max(),
                 отклонение=100 * (np.median(L_) - OPT_LEN) / OPT_LEN, время=budget))
df_tsp = pd.DataFrame(rows)
df_tsp.style.format({'лучший': '{:.0f}', 'медиана': '{:.0f}', 'худший': '{:.0f}', 'отклонение': '{:+.1f} %',
                     'время': '{:.1f} с'}).hide(axis='index')

Готово за 80 с; бюджет мультистарта 2-opt = 5.3 с на запуск


метод,запусков,лучший,медиана,худший,отклонение,время
ГА: CX + swap,10,772,818,847,+52.0 %,1.4 с
ГА: CX + инверсия,10,876,965,1071,+79.4 %,1.3 с
меметический ГА: CX + инверсия + 2-opt,5,538,538,538,+0.0 %,5.3 с
мультистарт 2-opt (без ГА),5,540,550,553,+2.2 %,5.3 с


In [12]:
# Лучший найденный тур против оптимального
best_run = min(ga_runs['меметический ГА: CX + инверсия + 2-opt'], key=lambda r: r['length'])
pure_run = min(ga_runs['ГА: CX + swap'], key=lambda r: r['length'])


def tour_trace(t, color, name, showlegend=True):
    loop = COORDS[np.r_[t, t[:1]]]
    return go.Scatter(x=loop[:, 0], y=loop[:, 1], mode='lines', name=name, showlegend=showlegend,
                      line=dict(color=color, width=2), hoverinfo='skip')


def city_trace(showlegend=False):
    return go.Scatter(x=COORDS[:, 0], y=COORDS[:, 1], mode='markers', name='города', showlegend=showlegend,
                      marker=dict(color=INK, size=6, line=dict(color=SURFACE, width=1)),
                      text=[f'город {i + 1}' for i in range(N_CITY)], hovertemplate='%{text}<br>(%{x}; %{y})<extra></extra>')


fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.03,
                    subplot_titles=[f'ГА: CX + swap — {pure_run["length"]:.0f}',
                                    f'меметический ГА — {best_run["length"]:.0f}', f'оптимальный тур TSPLIB — {OPT_LEN:.0f}'])
for c, (t, col) in enumerate([(pure_run['best'], PALETTE[1]), (best_run['best'], PALETTE[0]), (OPT_TOUR, PALETTE[2])], 1):
    fig.add_trace(tour_trace(t, col, '', showlegend=False), 1, c)
    fig.add_trace(city_trace(), 1, c)
    fig.update_xaxes(showticklabels=False, row=1, col=c)
    fig.update_yaxes(showticklabels=False, scaleanchor=f'x{c if c > 1 else ""}', row=1, col=c)
fig.update_layout(title='eil76: лучшие найденные маршруты и оптимальный', height=470, width=1300)
save(fig, 'B1_tours', width=1300, height=470)
fig.show()

In [13]:
# Сходимость (медиана по запускам) — ось x в числе вычислений длины тура, чтобы сравнение было честным
fig = go.Figure()
for k, (name, per_gen) in enumerate([('ГА: CX + swap', 150), ('ГА: CX + инверсия', 150),
                                      ('меметический ГА: CX + инверсия + 2-opt', 60)]):
    cur = np.median([r['hist_best'] for r in ga_runs[name]], axis=0)
    fig.add_trace(go.Scatter(x=np.arange(len(cur)) * per_gen + per_gen, y=cur, mode='lines', name=name,
                             line=dict(color=PALETTE[k], width=2.5)))
fig.add_hline(y=OPT_LEN, line=dict(color=GOAL, dash='dash', width=1.5), annotation_text='оптимум 538',
              annotation_position='bottom right', annotation_font_color=GOAL)
fig.update_layout(title='eil76: длина лучшего тура (медиана по запускам)', xaxis_title='число оценённых туров (log)',
                  xaxis_type='log', yaxis_title='длина тура', yaxis_type='log', height=440,
                  legend=dict(x=0.99, xanchor='right', y=0.99))
save(fig, 'B2_convergence', height=440)
fig.show()

In [14]:
# Анимация: лучший тур меметического ГА по поколениям (Старт / Пауза / Сброс)
steps = sorted(best_run['snaps'])
fig = go.Figure()
fig.add_trace(tour_trace(best_run['snaps'][0], PALETTE[0], 'лучший тур'))
fig.add_trace(city_trace(True))
fig.frames = [go.Frame(name=str(s), data=[tour_trace(best_run['snaps'][s], PALETTE[0], 'лучший тур')], traces=[0],
                       layout=go.Layout(title_text=f'Меметический ГА: поколение {s}, длина {tour_len(best_run["snaps"][s]):.0f}'))
              for s in steps]
fig.update_layout(title=f'Меметический ГА: поколение 0, длина {tour_len(best_run["snaps"][0]):.0f}', width=760, height=780,
                  margin=dict(t=110, b=120), yaxis=dict(scaleanchor='x'), **anim_controls(steps))
save(fig, 'B3_tour_animation', png=False)
fig.show()

## Таблицы для отчёта

In [15]:
t1 = df_r.copy()
t1 = pd.DataFrame({'Параметр': t1.параметр.str.replace('σ₀', '$\\sigma_0$').str.replace('BLX-α', 'BLX-$\\alpha$')
                   .str.replace('Pm', '$P_m$'), 'Значение': t1.значение.map(lambda v: f'{v:g}'),
                   'Медиана $f$': t1.медиана_f.map(lambda v: f'{v:.1e}'), 'Худшее $f$': t1.худшее_f.map(lambda v: f'{v:.2f}'),
                   'Успехов, \\%': t1.успехов.map(lambda v: f'{v:.0f}')})
to_latex(t1, 'tab_real_params', 'Вещественный ГА на функции Растригина, 50 поколений (20 запусков на значение)', 'tab:realparams', 'lcccc')
t2 = pd.DataFrame({'Функция': df_cmp.функция, 'Кодирование': df_cmp.кодирование,
                   'Медиана $f$': df_cmp.медиана_f.map(lambda v: f'{v:.1e}'), 'Лучшее $f$': df_cmp.лучшее_f.map(lambda v: f'{v:.1e}'),
                   'Худшее $f$': df_cmp.худшее_f.map(lambda v: f'{v:.1e}')})
to_latex(t2, 'tab_real_vs_binary', 'Вещественное и бинарное кодирование при равном бюджете (20 запусков)', 'tab:realbin', 'llccc')
short = {'ГА: CX + swap': 'ГА: CX + обмен', 'ГА: CX + инверсия': 'ГА: CX + инверсия',
         'меметический ГА: CX + инверсия + 2-opt': 'меметический ГА', 'мультистарт 2-opt (без ГА)': 'мультистарт 2-opt'}
t3 = pd.DataFrame({'Метод': df_tsp.метод.map(short), 'Запусков': df_tsp.запусков, 'Лучший': df_tsp.лучший.map(lambda v: f'{v:.0f}'),
                   'Медиана': df_tsp.медиана.map(lambda v: f'{v:.0f}'), 'Худший': df_tsp.худший.map(lambda v: f'{v:.0f}'),
                   'Отклонение, \\%': df_tsp.отклонение.map(lambda v: f'{v:+.1f}'), 'Время, с': df_tsp.время.map(lambda v: f'{v:.1f}')})
to_latex(t3, 'tab_tsp', 'Задача коммивояжёра eil76 (оптимум 538); меметический ГА~--- CX + инверсия + 2-opt; отклонение~--- по медиане', 'tab:tsp', 'lcccccc')
print(sorted(p.name for p in TABLES.glob('*.tex')))

['tab_real_params.tex', 'tab_real_vs_binary.tex', 'tab_tsp.tex']


## Выводы

**Часть А — вещественное кодирование**

1. Вещественный ГА (N = 60, BLX-0,5, гауссова мутация с убывающим шагом, турнир, 2 элиты) находит глобальный минимум функции Растригина до машинной точности ($f = 0$, $|x| \sim 10^{-9}$) за 150 поколений (9 060 вычислений) и обходит около 120 локальных минимумов. Популяция за 10 поколений собирается в центральной «яме» решётки.
2. **Параметры** (бюджет 50 поколений, по 20 запусков). Решающую роль играет мутация: без неё ($\sigma_0 = 0$ или $P_m = 0$) и со слишком малым шагом ($\sigma_0 = 0{,}02$) 20–30 % запусков застревают в локальном минимуме $f \approx 0{,}99$. Слишком частая мутация ($P_m = 0{,}5$) мешает доводке (успехов 80 %). BLX-α устойчив в диапазоне 0–0,5; при $\alpha = 1$ потомки разбрасываются слишком широко (95 %). Популяции из 20 особей мало (85 %), от 40 — надёжно.
3. **Вещественное кодирование против бинарного** при равном бюджете: на функции Шаффера F7 бинарный ГА упирается в предел своей сетки ($f = 0{,}020$), вещественный доходит до $f \approx 10^{-8}$ (медиана). На Растригине оба находят минимум, но вещественный точнее (медиана $f = 0$ против $10^{-8}$) и надёжнее (худший запуск бинарного ГА остановился в локальном минимуме $f = 1{,}2$). Вещественное представление не ограничено сеткой и естественно для непрерывных задач.

**Часть Б — задача коммивояжёра (eil76)**

4. **Исправлены данные:** в прежнем файле `eil76.tsp` координаты городов 59–76 не совпадали с TSPLIB. Эталонный набор проверен опубликованным оптимальным туром — ровно 538.
5. **ГА с циклическим кроссинговером** за 500 поколений (75 000 оценок) даёт туры 772–847 (медиана +52 % к оптимуму), и кривая ещё не вышла на плато. CX сохраняет абсолютные позиции городов, а для маршрута важна их **смежность**: потомок наследует позиции, но почти не наследует рёбра хороших родителей. Инверсия как мутация сама по себе хуже обмена (+79 %): в паре с CX она разрушает унаследованный порядок позиций.
6. **Меметический ГА** (CX + инверсия + 2-opt для каждого потомка) нашёл тур длины **538 — оптимум — во всех 5 запусках** за ~5 с. Найденный маршрут отличается от опубликованного, но имеет ту же длину: у eil76 несколько оптимальных туров. Мультистарт 2-opt без ГА за то же время даёт 540–553 (медиана +2,2 %), т.е. именно генетическое комбинирование локальных оптимумов доводит решение до глобального.